# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import os, getpass, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
ANCHOR = "DATE '2026-03-31'"

features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date >  {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END)        AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2
    HAVING imp_prev30 >= 100
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']
features = features.merge(
    con.sql(f"""SELECT content_hash_id, DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
                FROM {dim_content}""").df(),
    on='content_hash_id', how='left'
)
features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
print(f"{len(features):,} content items\n")

# Signal 1 — staleness, behind the refresh flag
features['age_bucket'] = pd.cut(features['content_age_days'],
    bins=[0, 90, 365, 730, np.inf], labels=['<90d', '90-365d', '1-2yr', '2yr+'])
print("Signal 1: content_age_days vs is_declining")
print(features.groupby('age_bucket', observed=True).agg(
    n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3))

# Signal 2 — prior visibility, behind quick-win logic
features['volume_bucket'] = pd.cut(features['imp_prev30'],
    bins=[0, 250, 1000, 5000, np.inf], labels=['100-250', '250-1000', '1000-5000', '5000+'])
print("\nSignal 2: imp_prev30 vs is_declining")
print(features.groupby('volume_bucket', observed=True).agg(
    n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3))

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82,549 content items

Signal 1: content_age_days vs is_declining
                n  decline_rate
age_bucket                     
<90d        16071         0.165
90-365d     53545         0.326
1-2yr       12933         0.271

Signal 2: imp_prev30 vs is_declining
                   n  decline_rate
volume_bucket                     
100-250        18877         0.335
250-1000       28594         0.301
1000-5000      25722         0.244
5000+           9356         0.260


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
stale = (features['content_age_days'] >= 365).astype(int)
visible = (features['imp_prev30'] >= 250).astype(int)
features['score'] = stale * visible * features['imp_prev30']
features['reason_code'] = np.where(features['score'] > 0, 'stale_but_visible', 'no_action_needed')
features['action'] = np.where(features['score'] > 0, 'review_for_refresh', 'no_action')

queue = features.sort_values('score', ascending=False).reset_index(drop=True)
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"{len(queue):,} rows written, {(queue['score']>0).sum():,} flagged for review")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = features['is_declining'].mean()
print(f"Base rate: {base_rate:.3f}   Precision@50: {precision_at_k(features['score'], features['is_declining'], 50):.3f}")

82,549 rows written, 9,823 flagged for review
Base rate: 0.286   Precision@50: 0.280


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top10 = queue.head(10)[['client_hash_id','content_hash_id','score','reason_code','action',
                          'content_age_days','imp_prev30','imp_last30','is_declining']]
top10

,client_hash_id,content_hash_id,score,reason_code,action,content_age_days,imp_prev30,imp_last30,is_declining
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,204896.0,stale_but_visible,review_for_refresh,410,204896.0,134264.0,1
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,198364.0,stale_but_visible,review_for_refresh,410,198364.0,124050.0,1
2,client_73cda7b4e4f265ea,content_e241d6415ac9e534,179842.0,stale_but_visible,review_for_refresh,412,179842.0,139537.0,1
3,client_e547b89c05043229,content_c9a0c2fdbdbfb562,158716.0,stale_but_visible,review_for_refresh,375,158716.0,60920.0,1
4,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,143570.0,stale_but_visible,review_for_refresh,410,143570.0,123242.0,0
5,client_e547b89c05043229,content_ec2e0346994fb5a5,139111.0,stale_but_visible,review_for_refresh,434,139111.0,232973.0,0
6,client_73cda7b4e4f265ea,content_cf651123f1085418,136959.0,stale_but_visible,review_for_refresh,412,136959.0,97720.0,1
7,client_73cda7b4e4f265ea,content_471d9cabce329a66,132856.0,stale_but_visible,review_for_refresh,375,132856.0,159378.0,0
8,client_73cda7b4e4f265ea,content_c9f840183215651b,126771.0,stale_but_visible,review_for_refresh,410,126771.0,19827.0,1
9,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,119767.0,stale_but_visible,review_for_refresh,410,119767.0,144254.0,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
print("Score inputs: content_age_days (static, dim_content) + imp_prev30 (pre-anchor window only)")
print("is_declining used only for review context below, never in the score")

disagreements = top10[top10['is_declining'] == 0]
print(f"{len(disagreements)} of top 10 are NOT actually declining per the label")
disagreements

Score inputs: content_age_days (static, dim_content) + imp_prev30 (pre-anchor window only)
is_declining used only for review context below, never in the score
4 of top 10 are NOT actually declining per the label


,client_hash_id,content_hash_id,score,reason_code,action,content_age_days,imp_prev30,imp_last30,is_declining
4,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,143570.0,stale_but_visible,review_for_refresh,410,143570.0,123242.0,0
5,client_e547b89c05043229,content_ec2e0346994fb5a5,139111.0,stale_but_visible,review_for_refresh,434,139111.0,232973.0,0
7,client_73cda7b4e4f265ea,content_471d9cabce329a66,132856.0,stale_but_visible,review_for_refresh,375,132856.0,159378.0,0
9,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,119767.0,stale_but_visible,review_for_refresh,410,119767.0,144254.0,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.